# CSA0812 – Assignment 5
## A2 – Design the Cleaning and Normalisation

**Student:** Darshan A  
**Register Number:** 1973260011  
**Course Outcome:** CO5  
**Marks:** 5 | K4 – Analyse

**Purpose:** This notebook is organised as a separate submission for this rubric item. Written explanations are provided in text cells. Where implementation is required, the Python cells can be run in Google Colab.

## A2.1 – Cleaning Rules (2 Marks)

Cleaning is performed after sorting readings by station, date and hour. The program keeps a cleaning log so that every changed or rejected value can be audited.

| Data problem | Rule | Justification |
|---|---|---|
| Missing value (`blank`, `NA`, `N/A`) | For a short gap, forward-fill from the previous valid reading of the same station and metric. If no previous value exists, use that station's median for the same metric and analysis period. If no valid median exists, leave it NULL and exclude it from that metric's aggregate. | A previous reading is a reasonable short-gap estimate; a station median is a robust fallback. Filling must never cross station boundaries. |
| Negative concentration or impossible reading | Treat it as invalid, replace with missing and apply the missing-value rule; if it exceeds the accepted sensor limit, log the row as a sensor fault. | Negative PM/NO₂ values are not physically meaningful concentration readings. Very large values may be sensor faults. |
| Duplicate row | Remove exact duplicate rows. For multiple conflicting rows with the same station/date/hour, retain the first valid row according to the stated policy and log the conflict for review. | Prevents the same observation from influencing the mean more than once. |
| Fahrenheit temperature | Convert to Celsius using `C = (F - 32) × 5/9` before statistics or charts. | All stations must use one common unit for comparison. |
| Invalid date/hour | Reject and log a row if the date cannot be parsed or hour is not an integer from 0 to 23. | Invalid time values cannot be reliably grouped by day or hour. |
| Inconsistent station label | Trim whitespace and standardise case/title format; map aliases to a controlled station name. | Prevents one station appearing as multiple categories. |

**Important implementation decision:** Imputation is done separately for each station and pollutant. It must not use future data to fill a past value in a real-time system unless an offline retrospective analysis is explicitly being performed.

## A2.2 – Normalisation (2 Marks)

Normalisation is applied to cleaned PM2.5 values. The fitted minimum, maximum, mean and standard deviation should be calculated from a defined reference dataset and reused when later data is transformed.

### Min–max normalisation

Formula:

`x_scaled = (x - x_min) / (x_max - x_min)`

This maps the reference minimum to 0 and maximum to 1. It is appropriate when a fixed bounded range is useful, such as comparing several features on a common scale. It is sensitive to extreme outliers because one unusually large reading can compress most normal values into a narrow interval.

If `x_max = x_min`, set all scaled values to 0 to avoid division by zero.

### Z-score standardisation

Formula:

`z = (x - mean) / standard_deviation`

This expresses each value as the number of standard deviations above or below the mean. It is useful when a model or comparison expects approximately centred, scale-adjusted values and when the distance from the mean is meaningful. It is not bounded between 0 and 1 and may be affected by outliers.

If standard deviation is zero, return 0 for every value.

### Comparison

| Method | Output range | Appropriate use | Main caution |
|---|---|---|---|
| Min–max | Usually 0 to 1 on the fit data | Bounded feature scaling and display | Outliers strongly affect min/max |
| Z-score | Unbounded; mean 0 and SD 1 on fit data | Comparing deviations and many statistical/ML methods | Outliers affect mean and SD |

Normalisation does not correct bad sensor readings. Validation and cleaning must happen first.

## A2.3 – Formatting (1 Mark)

The cleaned dataset uses consistent formats:

- **Dates:** ISO 8601 date text `YYYY-MM-DD`, for example `2026-10-03`.
- **Time:** Integer hour from 0 to 23; optionally derive an ISO timestamp for the database.
- **Station names:** Trim leading/trailing whitespace, standardise to one canonical station name, for example `Adyar` and `T Nagar`.
- **Numbers:** Store pollutant readings and temperature as numeric values (REAL/float), not formatted strings. Keep a consistent decimal precision for exported CSV, such as one decimal place.
- **Temperature:** Store the cleaned analytical value in Celsius, with the original station unit retained separately for audit.
- **Missing data:** Store as SQL NULL or an empty field in a CSV export; do not store text `"NA"` in a numeric column.
- **Column names:** Lowercase snake_case, such as `station`, `observation_date`, `pm25`, `temperature_c`.

## Alternative Concept and Complexity Analysis

If missing readings were dropped instead of imputed, only observations with actual valid sensor values would contribute to the mean. This avoids introducing estimated values, but daily averages may become biased if missingness is not random. For example, if a sensor fails during high-pollution periods, dropping those readings may make the daily average appear lower than it really was.

| Method | Processing | Time complexity | Extra space |
|---|---|---|---|
| Drop missing values | Check every reading and retain valid values | O(n) | O(1) additional space if streamed; O(n) if retained in a new array |
| Forward fill | Keep the previous valid value for each station/metric and scan rows once | O(n) | O(stations × metrics) state, plus output if retained |
| Median fill | Find each station/metric median, then replace missing values | O(n log n) if values are sorted to calculate medians; O(n) for a selection-based median algorithm | O(n) if values are collected for median calculation |

**Interpretation:** Dropping is fastest and preserves only measured values, but it reduces the sample count and can distort daily averages. Forward fill is efficient for short gaps in time-series data. Median fill is more robust to outliers than mean fill but costs additional work and should be used carefully. The chosen policy should be recorded so analysts can distinguish measured readings from imputed readings.

## A2 – Conclusion

The cleaning rules handle missing readings, sensor faults, duplicate observations, Fahrenheit conversion and inconsistent formats. Min–max scaling is suitable for a bounded 0–1 feature, while z-score standardisation expresses deviations from the mean. All transformations are applied only after validating the raw readings.